### Brief Intro of 4 Advanced CNN models for Image Segmentation

### 1. U-Net
- **Introduction**:

    U-Net, introduced in 2015 for biomedical imaging, uses a U-shaped encoder-decoder architecture with skip connections to enable precise pixel-level segmentation even with limited data.​

 - **Architecture**:

    The contracting path (encoder) downsamples via convolutions and max-pooling to capture context. The expanding path (decoder) upsamples with transposed convolutions, fusing low-level details from skip connections to preserve boundaries.​

- **Pros**:

    Excellent localization, data-efficient, fast inference.

- **Cons**:

    Limited multi-scale context, may struggle with complex scenes.​

- **Advantages over Simple CNN**:

    Unlike basic CNNs with fixed receptive fields and no upsampling, U-Net's skip connections recover spatial details lost in downsampling, yielding sharper boundaries and higher mIoU (e.g., outperforms sliding-window CNNs by incorporating global context).

In [1]:
# U-Net implementation in python
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = nn.MaxPool2d(2)
        self.down2 = nn.MaxPool2d(2)
        self.down3 = nn.MaxPool2d(2)
        self.down4 = nn.MaxPool2d(2)
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.outc = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.inc(self.down1(x1))
        x3 = self.inc(self.down2(x2))
        x4 = self.inc(self.down3(x3))
        x5 = self.inc(self.down4(x4))
        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.inc(x)
        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.inc(x)
        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.inc(x)
        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.inc(x)
        return self.outc(x)


### 2. DeepLab
- **Introduction**:

    DeepLab (v3+), from Google, excels in semantic segmentation using atrous convolutions for dense feature extraction across scales, ideal for dense prediction tasks like autonomous driving.​

- **Architecture**:

    Backbone (e.g., ResNet) extracts features, followed by Atrous Spatial Pyramid Pooling (ASPP) with parallel dilated convolutions at multiple rates. Decoder refines boundaries via bilinear upsampling and fusions.​

- **Pros**:
    
    Multi-scale context, sharp boundaries via CRF, state-of-the-art mIoU.
- **Cons**:
    
    Computationally heavier, complex hyperparameters.​

- **Advantages over Simple CNN**:

    Atrous convolutions expand receptive fields without resolution loss (unlike CNN pooling), capturing multi-scale context; ASPP outperforms CNN's fixed kernels on PASCAL VOC (79.7% mIoU vs lower)

In [2]:
## DeepLab implementation in python

import torch
import torch.nn as nn
from torchvision.models.segmentation import deeplabv3_resnet50

# Simple DeepLabv3+ using torchvision (pretrained available)
model = deeplabv3_resnet50(pretrained=True, num_classes=21)
# For custom: model.classifier[4] = nn.Conv2d(256, num_classes, 1)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth


100%|██████████| 161M/161M [00:01<00:00, 109MB/s]


#### 3. PSPNet
- **Introduction**:

    PSPNet addresses scene parsing with pyramid pooling to aggregate global context, winning ImageNet Scene Parsing Challenge.​

- **Architecture**:

    ResNet backbone with dilated convolutions, followed by Pyramid Pooling Module (pooling at 1x1, 2x2, 3x3, 6x6 scales, concatenated), then final convolution.​

- **Pros**:
    Rich global scene context, high accuracy on complex scenes.
- **Cons**:
    Higher memory use from pyramid module, slower than U-Net.​

- **Advantages over Simple CNN**:

    Pyramid pooling captures multi-scale context missing in CNN's local receptive fields, boosting mIoU to 91%+ on datasets like COCO vs basic CNN baselines.

In [3]:
## PSPNet Implementation using python

import torch
import torch.nn as nn
import torch.nn.functional as F

class PyramidPooling(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool1 = nn.AdaptiveAvgPool2d(1)
        self.pool2 = nn.AdaptiveAvgPool2d(2)
        self.pool3 = nn.AdaptiveAvgPool2d(3)
        self.pool6 = nn.AdaptiveAvgPool2d(6)
        self.conv1 = nn.Conv2d(in_channels, out_channels//4, 1)
        self.conv2 = nn.Conv2d(in_channels, out_channels//4, 1)
        self.conv3 = nn.Conv2d(in_channels, out_channels//4, 1)
        self.conv6 = nn.Conv2d(in_channels, out_channels//4, 1)

    def forward(self, x):
        h, w = x.size(2), x.size(3)
        p1 = F.interpolate(self.conv1(self.pool1(x)), (h,w))
        p2 = F.interpolate(self.conv2(self.pool2(x)), (h,w))
        p3 = F.interpolate(self.conv3(self.pool3(x)), (h,w))
        p6 = F.interpolate(self.conv6(self.pool6(x)), (h,w))
        return torch.cat([p1,p2,p3,p6,x], dim=1)

# Simplified PSPNet (backbone + PSP + final conv)
class PSPNet(nn.Module):
    def __init__(self, n_classes=21):
        super().__init__()
        # Assume resnet backbone as encoder (from torchvision)
        self.encoder = ...  # ResNet50 dilated
        self.psp = PyramidPooling(2048, 512)
        self.final = nn.Conv2d(2560, n_classes, 1)  # 2048+512

    def forward(self, x):
        feat = self.encoder(x)  # Get final feature map
        psp_feat = self.psp(feat)
        return self.final(psp_feat)


### 4. SegFormer

- **Introduction**:

    SegFormer, introduced by NVIDIA in 2021, combines hierarchical transformers with a simple MLP decoder to outperform CNN-based models on benchmarks like ADE20K (50.3% mIoU for B4 variant) while using fewer parameters.

- **Architecture**:

    A hierarchical Transformer encoder (MiT - Mix Transformer) processes images at four scales with overlapping patch embeddings, efficient self-attention, and feed-forward blocks—no positional encodings needed. The all-MLP decoder fuses multi-scale features via linear layers and upsampling for final segmentation.

- **Pros**:

    Resolution-agnostic (no positional interpolation issues), lightweight decoder (5x smaller than prior SOTA), excellent speed-accuracy trade-off.

- **Cons**:
   
    Transformer-heavy compute on very high-res images, less interpretable than CNNs.​

- **Advantages over Simple CNN**:

    Unlike simple CNNs with fixed receptive fields and pooling losses, SegFormer's transformers capture global dependencies natively across scales; it beats U-Net/DeepLab baselines with better zero-shot robustness and no complex modules like ASPP or skip connections.

In [4]:
## SegFormer Implementation using python

import torch
import torch.nn as nn
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

# Using HuggingFace (pretrained MiT-B0 to B5)
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512", num_labels=150
)
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")

# Forward pass
# inputs = processor(images=image, return_tensors="pt")
# outputs = model(**inputs)
# logits = outputs.logits  # Shape: [1, num_classes, H/4, W/4]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:417: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
